<H1>DOCUMENT LOADER</H1>

In [1]:
from langchain_community.document_loaders import Docx2txtLoader
import os
def document_loader(folder_path):

    document = []

    for file_name in os.listdir(folder_path):
        if file_name.endswith('.docx'):
            file_path = os.path.join(folder_path,file_name)

            documentLoader = Docx2txtLoader(file_path)
            pages = documentLoader.load()

            document.extend(pages)

    return document


C:\Users\Ahmad\AppData\Local\Temp\ipykernel_9804\2373767959.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


<h1>Text splitter</h1>

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def text_splitter(chunks):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap  = 50
    )
    chunk = splitter.split_documents(chunks)

    return chunk

<h1>Vector database</h1>

In [3]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

def create_vector_db(chunk):
        embedding = HuggingFaceEmbeddings(model = 'sentence-transformers/all-MiniLM-L6-v2')

        vector_db = FAISS.from_documents(
                chunk,
                embedding
        )


        return vector_db


<h1>KEYWORD SEARCH</h1>

In [4]:
from rank_bm25 import BM25Okapi

class BM250_Search:
    def __init__(self,chunks):
        self.chunks = chunks

        tokenized_data = []

        for x in chunks:
            tokens = x.page_content.lower().split()
            tokenized_data.append(tokens)

        self.bm250 = BM25Okapi(tokenized_data) 

    def search(self,question,k=5):

            query = question.lower().split()
            scores = self.bm250.get_scores(query)

            ranked_answer = sorted(range(len(scores)),
                                key = lambda index: scores[index],
                                reverse = True)

            top_k = ranked_answer[:k]

            result = []

            for r in top_k:
                result.append(self.chunks[r])

            return result


<h1>Hybrid search</h1>

In [5]:
class Hybrid_Search:
    def __init__(self,vector_db,b250_db):
        self.vector_db = vector_db
        self.b250_db = b250_db


    def combinedResult(self,query,k=5):

        vector_result = self.vector_db.similarity_search(query,k=k)
        bm250_result = self.b250_db.search(query,k=k)

        combined_result = vector_result + bm250_result

        unique_result = []
        unseen_result = set()


        for x in combined_result:
            if x.page_content not in unseen_result:
                unique_result.append(x)
                unseen_result.add(x.page_content)
        return unique_result


<h1>Reranker</h1>

In [6]:
from sentence_transformers import CrossEncoder

class ReRanker:
    def __init__(self):
       self.model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    def rerank(self,query,document,k=5):

        pairs = []

        for c in document:
            pairs.append([
                query,
                c.page_content
            ])
        scores = self.model.predict(pairs)

        rerank_answers = sorted(zip(scores,document),
                                key = lambda i:i[0],
                                reverse = True)

        top_answer = rerank_answers[:k]

        result = []

        for i,document in top_answer:
            result.append(document)

        return result


<h1>Prompt</h1>

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

def generate_answer(query,document):

    context = ""
    for d in document:
        context += d.page_content
        context += "\n\n"

    prompt = ChatPromptTemplate.from_template(
        '''
        Answer the query from the context.
        if the query are out of the context , just say : "I dont know the answer"

        context
        {context}

        query
        {query}
        
        '''
    )

    llm_model = ChatGroq(
        model = 'openai/gpt-oss-20b',
        temperature = 0
    )

    chain = prompt | llm_model

    response = chain.invoke({
        'context':context,
        'query':query
    })

    return response.content

<h1>PIPELINE</h1>

In [10]:
import gradio as gr

#document loader
folder_path = 'E:\GEN-AI-PROJECTS'
document = document_loader(folder_path)

# chunking
chunk = text_splitter(document)

#create vector_db
vector_db = create_vector_db(chunk)

#key word search
bm250 = BM250_Search(chunk)

#hybrid search
hybrid = Hybrid_Search(vector_db,bm250)

# rerank
rerank = ReRanker()

# generate answers

def Generate_Answers(query,k=5):

    retrived_doc = hybrid.combinedResult(query,k=5)

    ranked_final = rerank.rerank(query,retrived_doc,k=5)

    answer = generate_answer(query,ranked_final)

    source = ""

    for i,doc in enumerate(retrived_doc):
        file_source = doc.metadata.get(
            'source',
            'unknown'
        )

        source += f'\nSource {i+1}: Page {file_source}'

    final_response = answer
    final_response += "\n\nSource: "
    final_response += source

    return final_response

demo = gr.Interface(
    fn = Generate_Answers,
    inputs = gr.Textbox(
        label = 'Enter the question'
    ),
    outputs = gr.Textbox(
        label = 'Answer',
        lines = 15
    ),
    title = 'DEEP KNOWLEDGE RAG',


)


demo.launch(share=True)




Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7862

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
